# Cyber Attack Modeling via Hawkes Processes

## Objective

We model cyber attack events using Hawkes processes to capture:

- Self-excitation
- Temporal clustering
- Burst dynamics

We compare:

- Single Hawkes (K=1)
- Bimodal Hawkes (K=2)

We further analyze uncertainty via bootstrap approximation.

## 1. Setup & Core Model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

from src.simulator import simulate_hawkes
from src.inference import branching_em, bimodal_branching_em, compute_bic

## 2. Data Loading

In [ ]:
# ── Empirical Data Loading and Preprocessing ─────────────────────────
# Dynamically locate and parse the network traffic log.
# This routine implements robust parsing across multiple text encodings
# and isolates the continuous temporal sequence for point process modeling.

DATA_PATH = "data"

files = [f for f in os.listdir(DATA_PATH) if f.endswith(".csv")]

if not files:
    raise FileNotFoundError("Critical Error: No CSV datasets located in the target directory.")

file_path = os.path.join(DATA_PATH, files[0])
print(f"Target empirical dataset: {files[0]}")

# Attempt robust parsing using common character encodings
encodings = ["utf-8", "utf-8-sig", "latin1", "cp1252", "gbk"]
df = None

for enc in encodings:
    try:
        df = pd.read_csv(file_path, encoding=enc, low_memory=False)
        print(f"Successfully parsed using encoding: {enc}")
        break
    except Exception:
        continue

if df is None:
    raise ValueError("Decoding Error: Dataset could not be parsed with standard encodings.")

# Isolate the temporal sequence (identifying the first strictly numeric column)
events = None
for col in df.columns:
    try:
        vals = df[col].astype(float)
        events = np.sort(vals.values)
        print(f"Temporal feature identified: '{col}'")
        break
    except Exception:
        continue

if events is None:
    raise ValueError("Format Error: No numeric temporal feature located in the dataset.")

# Data sanitization: enforce finite, non-negative timestamps
events = events[np.isfinite(events)]
events = events[events >= 0]

if len(events) < 10:
    raise ValueError("Data Insufficiency: The temporal sequence contains fewer than 10 valid events.")

T = float(events.max())

print("-" * 50)
print(f"Total extracted events (N) : {len(events)}")
print(f"Observation horizon (T)    : {T:.2f}")

## 3. Model Fitting

In [ ]:
mu, alpha, beta, ll1, _ = branching_em(events, T)

ll1_val = ll1[-1]
bic1 = compute_bic(ll1_val, 3, len(events))

print("mu:", mu)
print("alpha:", alpha)
print("beta:", beta)
print("BIC:", bic1)

## 4. Visualize Results

In [ ]:
plt.plot(ll1)
plt.title("Log-Likelihood Convergence")
plt.xlabel("Iteration")
plt.ylabel("LL")
plt.show()
plt.figure(figsize=(10,2))
plt.eventplot(events)
plt.title("Event Timeline")
plt.show()
def intensity(t, events, mu, alpha, beta):
    return mu + np.sum(alpha*np.exp(-beta*(t-events[events<t])))

grid = np.linspace(0,T,500)
lam = [intensity(t,events,mu,alpha,beta) for t in grid]

plt.figure(figsize=(10,4))
plt.plot(grid,lam)
plt.title("Intensity Function")
plt.show()

### 4.1 Branching Heatmap

In [ ]:
def branch_matrix(events, mu, alpha, beta):
    n = len(events)
    dt = events[:,None] - events[None,:]
    mask = dt > 0

    L = np.zeros((n,n))
    L[mask] = alpha*np.exp(-beta*dt[mask])
    np.fill_diagonal(L, mu)

    lam = L.sum(axis=1)
    P = L/lam[:,None]

    return P

P = branch_matrix(events, mu, alpha, beta)

plt.imshow(P, aspect='auto')
plt.colorbar()
plt.title("Branching Structure")
plt.show()

### 4.2. Inter-event

In [ ]:
inter = np.diff(events)

plt.hist(inter, bins=30, density=True, alpha=0.6)

x = np.linspace(0,inter.max(),100)
lam_est = 1/np.mean(inter)
plt.plot(x, lam_est*np.exp(-lam_est*x))

plt.title("Inter-event Time Distribution")
plt.show()

## 5. Bootstrap Bayesian

In [ ]:
import sys
sys.path.append('.')  

from src.simulator import simulate_hawkes
from src.inference import bimodal_branching_em

# ── Parametric Bootstrap for Parameter Uncertainty ───────────────────
#
# For a Hawkes process, the standard nonparametric bootstrap
# (resampling event times) is invalid: it breaks the self-excitation
# structure that the model is built on.
#
# The correct approach is parametric bootstrap:
#   1.  Fit the model once to the observed data  →  (mu_hat, alpha_hat, beta_hat)
#   2.  Simulate B new realisations from that fitted model
#   3.  Refit the model on each simulated realisation
#   4.  The spread of the B estimates approximates sampling uncertainty
#
# This is equivalent to sampling from the asymptotic distribution of
# the MLE, and converges to the true Fisher-information-based CI as
# T → ∞.
# ─────────────────────────────────────────────────────────────────────

def parametric_bootstrap(mu_hat, alpha_hat, beta_hat, T, B=200, seed=0):
    """
    Parametric bootstrap for Hawkes process parameter uncertainty.

    Parameters
    ----------
    mu_hat, alpha_hat, beta_hat : MLE estimates from observed data
    T    : original observation horizon
    B    : number of bootstrap replicates
    seed : base random seed for reproducibility

    Returns
    -------
    boot : ndarray of shape (B, 3) — columns are mu, alpha, beta
    """
    boot = []
    for b in range(B):
        rng_b = np.random.default_rng(seed + b)
        sim   = simulate_hawkes(mu_hat, alpha_hat, beta_hat, T, rng_b)
        if len(sim) < 10:
            continue
        try:
            m, a, bb, _, _ = branching_em(sim, T)
            if np.isfinite([m, a, bb]).all():
                boot.append([m, a, bb])
        except Exception:
            continue
    return np.array(boot)


rng_boot = np.random.default_rng(0)
boot = parametric_bootstrap(mu, alpha, beta, T, B=200, seed=0)

print(f"Bootstrap replicates completed: {len(boot)}")
print(f"{'Parameter':<12} {'Estimate':>10} {'Mean':>10} {'Std':>8} {'2.5%':>8} {'97.5%':>8}")
print("-" * 58)
for k, (name, est) in enumerate(zip(['mu', 'alpha', 'beta'], [mu, alpha, beta])):
    vals = boot[:, k]
    print(f"{name:<12} {est:10.4f} {vals.mean():10.4f} "
          f"{vals.std():8.4f} {np.percentile(vals,2.5):8.4f} "
          f"{np.percentile(vals,97.5):8.4f}")

# Plot bootstrap distributions
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
names  = [r'$\mu$', r'$\alpha$', r'$\beta$']
estims = [mu, alpha, beta]
colors = ['#2C6FAC', '#E05A2B', '#3BA35A']

for k, (ax, name, est, col) in enumerate(zip(axes, names, estims, colors)):
    ax.hist(boot[:, k], bins=25, color=col, alpha=0.75, edgecolor='white',
            linewidth=0.5, density=True)
    ax.axvline(est, color='black', linewidth=1.8, linestyle='-',
               label=f'MLE = {est:.3f}')
    ax.axvline(np.percentile(boot[:, k], 2.5),  color='black',
               linewidth=1.0, linestyle='--', alpha=0.6)
    ax.axvline(np.percentile(boot[:, k], 97.5), color='black',
               linewidth=1.0, linestyle='--', alpha=0.6,
               label='95% CI')
    ax.set_xlabel(name, fontsize=12)
    ax.set_ylabel('Density', fontsize=11)
    ax.set_title(f'Bootstrap distribution of {name}', fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Parametric Bootstrap: Parameter Uncertainty Quantification',
             fontsize=12)
plt.tight_layout()
plt.show()

## 6. Model Comparison: K=1 vs K=2 via BIC

In [ ]:
# ── Section 6: Model Comparison — K=1 vs K=2 via BIC ─────────────────

# K=1 model: already fitted in Section 3
ll_k1  = ll1[-1]
bic_k1 = compute_bic(ll_k1, 3, len(events))


def bimodal_ll_fast(events, T, mu2, alpha2, beta2):
    """
    O(N) log-likelihood for the two-component mixture model.

    Uses the same Markovian recursion as the single-component case,
    applied independently to each component and summed:

        lambda_total(t_i) = sum_k [ mu_k + alpha_k * A_i^(k) ]

    where  A_i^(k) = exp(-beta_k * (t_i - t_{i-1})) * (1 + A_{i-1}^(k)).

    This is O(N) in both time and space, compared to O(N²) for the
    naive pairwise approach.
    """
    n         = len(events)
    lam_total = np.zeros(n)

    for k in range(2):
        A = np.zeros(n)
        for i in range(1, n):
            A[i] = (np.exp(-beta2[k] * (events[i] - events[i-1]))
                    * (1 + A[i-1]))
        lam_total += mu2[k] + alpha2[k] * A

    lam_total = np.maximum(lam_total, 1e-300)
    comp = sum(
        mu2[k] * T
        + (alpha2[k] / beta2[k]) * np.sum(1 - np.exp(-beta2[k] * (T - events)))
        for k in range(2)
    )
    return float(np.sum(np.log(lam_total)) - comp)


try:
    est_mu2, est_alpha2, est_beta2, R_bg2, R_trig2 = bimodal_branching_em(
        events, T
    )

    ll_k2  = bimodal_ll_fast(events, T, est_mu2, est_alpha2, est_beta2)
    bic_k2 = compute_bic(ll_k2, 7, len(events))

    print("\n── Model Comparison ──────────────────────────────────")
    print(f"  K=1  |  log L = {ll_k1:10.2f}  |  BIC = {bic_k1:10.2f}")
    print(f"  K=2  |  log L = {ll_k2:10.2f}  |  BIC = {bic_k2:10.2f}")
    print(f"  ΔBIC (K1 − K2) = {bic_k1 - bic_k2:.2f}")
    if bic_k2 < bic_k1:
        print("  → K=2 preferred by BIC")
    else:
        print("  → K=1 preferred by BIC (K=2 complexity not justified)")
    print("──────────────────────────────────────────────────────")

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    axes[0].bar(['K=1 (3 params)', 'K=2 (7 params)'],
                [bic_k1, bic_k2],
                color=['#2C6FAC', '#E05A2B'], alpha=0.85, edgecolor='white')
    axes[0].set_ylabel('BIC  (lower = better)')
    axes[0].set_title('Model Selection via BIC')
    axes[0].grid(True, axis='y', alpha=0.3)

    axes[1].bar(['K=1', 'K=2'], [ll_k1, ll_k2],
                color=['#2C6FAC', '#E05A2B'], alpha=0.85, edgecolor='white')
    axes[1].set_ylabel('Log-likelihood  (higher = better)')
    axes[1].set_title('Log-likelihood Comparison')
    axes[1].grid(True, axis='y', alpha=0.3)

    plt.suptitle('K=1 vs K=2 Hawkes Model Comparison', fontsize=12)
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f"K=2 fit failed: {e}")
    print(f"K=1 only  —  BIC = {bic_k1:.2f}  (log L = {ll_k1:.2f})")

## 7. Strategic Interpretation: Branching Ratio and Attack Dynamics

In the context of the self-exciting Hawkes process, the **Branching Ratio ($\eta = \alpha / \beta$)** serves as a critical dimensionless metric for assessing the stability and "infectivity" of the observed cyber attack sequence. 

### 7.1 Mathematical Significance
The parameter $\eta$ represents the expected number of direct offspring events triggered by a single ancestor event. 
- **Sub-criticality ($\eta < 1$):** The attack process is stable; cascades eventually die out.
- **Criticality ($\eta \to 1$):** The system approaches a "tipping point" where a single probe could trigger an infinite chain of events.

### 7.2 Cybersecurity Domain Mapping
- **Profile 1: High Background Rate, Low $\eta$ (Automated Scanning)**
  If $\mu$ is high but $\eta \approx 0.1$, the data reflects "background noise" from broad-spectrum botnets. These attacks are independent, non-coordinated, and arrive at a constant rate regardless of past activity.
  
- **Profile 2: Low Background Rate, High $\eta$ (Targeted APT Probing)**
  If $\mu$ is low but $\eta > 0.4$, this indicates a "stealthy but aggressive" actor. While initial entry attempts are rare, a single successful probe triggers a rapid, endogenous sequence of lateral movement or data exfiltration.

### 7.3 Policy Implications
Quantifying $\eta$ allows for **proactive resource allocation**. High-infectivity periods ($\eta \uparrow$) require immediate automated blocking of IP subnets to break the self-exciting cycle, whereas high-background periods ($\mu \uparrow$) suggest the need for updated firewall blacklists.

In [ ]:
# ── Quantitative Security Metric: Branching Ratio Analysis ───────────

# K=1 model infectivity
branching_ratio = alpha / beta

print("── Security Metric Report ──────────────────────────────────")
print(f"  K=1  Branching Ratio (η = α/β) : {branching_ratio:.4f}")

if branching_ratio > 0.5:
    insight = "High Infectivity — each probe likely triggers follow-on attacks (APT)"
elif branching_ratio > 0.2:
    insight = "Moderate Self-Excitation — mixed scanning and coordinated probing"
else:
    insight = "Low Infectivity — attacks appear largely independent (botnet scanning)"

print(f"       Operational Insight        : {insight}")
print("-" * 60)

# K=2 component-level analysis
# Only executes if bimodal_branching_em succeeded in Section 6
try:
    print("  K=2  Component-level branching ratios:")
    for k, (a_k, b_k) in enumerate(zip(est_alpha2, est_beta2)):
        eta_k = a_k / b_k
        if eta_k > 0.4:
            tag = "APT lateral movement profile"
        elif eta_k > 0.15:
            tag = "Mixed / transitional profile"
        else:
            tag = "Automated scanning profile"
        print(f"       Component {k+1}  η_{k+1} = {eta_k:.4f}  →  {tag}")
except NameError:
    print("  K=2 parameters not available "
          "(bimodal fitting may have failed in Section 6).")

print("────────────────────────────────────────────────────────────")

## References

1. **Hawkes, A. G. (1971).** "Spectra of some self-exciting and mutually exciting point processes." *Biometrika*, 58(1), 83–90.  
   *(The foundational paper for the self-exciting model used in Section 1.)*

2. **Dempster, A. P., Laird, N. M., & Rubin, D. B. (1977).** "Maximum Likelihood from Incomplete Data via the EM Algorithm." *Journal of the Royal Statistical Society: Series B*, 39(1), 1–38.  
   *(The theoretical basis for the monotonic convergence of the EM algorithm shown in Section 3.)*

3. **Ogata, Y. (1981).** "On Lewis' simulation method for point processes." *IEEE Transactions on Information Theory*, 27(1), 23-31.  
   *(Standard citation for the thinning algorithm used in your `simulator.py`.)*

4. **Zhuang, J., Ogata, Y., & Vere-Jones, D. (2002).** "Stochastic declustering of space-time earthquake occurrences." *Journal of the American Statistical Association*, 97(458), 369–380.  
   *(CRITICAL: Your `branching_em` implementation is technically a "Stochastic Declustering" method, first formalized here.)*